# 마켓컬리 간편식·밀키트·샐러드 이미지 수집

**목표: 간편식·밀키트·샐러드 50상품 × 대표 1장 + 상세 1장 = 총 100장**

Chrome이 설치된 PC에서 실행할 수 있음.

- **Selenium**: 상품 페이지를 열고 이미지가 나타날 때까지 기다립니다.
- **BeautifulSoup**: 상품명, 이미지 주소, 원래 alt를 가져옵니다.
- **requests**: 이미지 파일을 내려받습니다.


**새로 시험 수집하려면 3번 셀에서 `SAVE_ROOT = Path("new_run")`, `TARGET_PRODUCTS = 3`으로 바꾸세요.**

## 1. 패키지 설치
처음 한 번 실행. 설치 후에도 import 오류가 나면 노트북 커널을 다시 시작.


In [15]:
# !pip install "selenium>=4.11,<5" beautifulsoup4 requests pandas pillow


## 2. 필요한 도구 불러오기


In [16]:
import time
from datetime import datetime, timezone, timedelta
from io import BytesIO
from pathlib import Path
from urllib.parse import urljoin, urlsplit, unquote

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


## 3. 카테고리와 저장 위치 설정

- 카테고리명은 폴더명·CSV 이름·CSV의 `category` 값에 동일하게 적용.
- `SAVE_ROOT`는 카테고리 폴더를 만들 상위 위치. `Path.cwd()`는 현재 작업 폴더.
- 같은 출력 폴더로 다시 실행하면 기존 CSV에 기록된 상품은 건너뜀.


In [17]:
CATEGORY = "convenience_meals"
CATEGORY_URL = "https://www.kurly.com/categories/912"
TARGET_PRODUCTS = 50
SAVE_ROOT = Path.cwd()  # 새로 수집하려면 Path("new_run")으로 변경

category_dir = SAVE_ROOT / CATEGORY
image_dir = category_dir / "images"
metadata_path = category_dir / f"{CATEGORY}_metadata.csv"
image_dir.mkdir(parents=True, exist_ok=True)

columns = [
    "product_id", "product_name", "category", "product_url", "image_id",
    "image_position", "image_url", "original_alt", "alt_exists", "crawl_date"
]

rows = []
completed_ids = set()
if metadata_path.exists():
    previous_data = pd.read_csv(metadata_path, dtype={"product_id": str}, keep_default_na=False)
    rows = previous_data.to_dict("records")
    completed_ids = set(previous_data["product_id"])
    if len(rows) != len(completed_ids) * 2:
        raise ValueError("기존 CSV에 상품당 2행이 있는지 확인해 주세요.")

print("저장 위치:", category_dir.resolve())
print("기존 데이터:", len(completed_ids), "상품 /", len(rows), "장")


저장 위치: D:\2026\Capstone2\Capstone-Design-Alt-Text\convenience_meals
기존 데이터: 50 상품 / 100 장


## 4. 반복해서 쓰는 함수

이미지 주소를 읽는 함수, 중복 주소를 비교하는 함수, 이미지를 내려받는 함수, CSV를 저장하는 함수.
파일 확장자는 다운로드한 실제 이미지 형식에 맞춤.


In [18]:
def get_image_url(image_tag, page_url):
    for attribute in ["src", "data-src"]:
        address = image_tag.get(attribute, "").strip()
        if address and not address.startswith(("data:", "blob:")):
            return urljoin(page_url, address)
    raise ValueError("이미지 주소를 찾지 못했습니다.")


def image_key(image_url):
    # 같은 이미지의 크기만 다른 CDN 주소를 비교.
    path = unquote(urlsplit(image_url).path)
    return path.split("/src/")[-1]


def download_image(image_url):
    response = requests.get(image_url, timeout=30)
    if response.status_code in [401, 403, 429]:
        raise PermissionError("접근 또는 요청 제한으로 수집을 중단합니다.")
    response.raise_for_status()

    with Image.open(BytesIO(response.content)) as image:
        extension = image.format.lower()
        image.verify()
    if extension == "jpeg":
        extension = "jpg"
    if extension not in ["jpg", "png", "webp", "gif"]:
        raise ValueError("지원하지 않는 이미지 형식입니다.")
    return response.content, extension


import re
import pandas as pd

def save_metadata():
    # 기존 데이터와 새 데이터 모두 alt_exists를 다시 계산
    for row in rows:
        value = row.get("original_alt")
        text = "" if pd.isna(value) else str(value)

        # 공백·줄바꿈·일부 보이지 않는 문자만 있으면 없는 것으로 판정
        text = re.sub(r"[\s\u200b-\u200f\u2060\ufeff]", "", text)
        row["alt_exists"] = int(bool(text))

    # original_alt 원본은 유지하고 alt_exists만 수정해서 저장
    data = pd.DataFrame(rows, columns=columns)
    data.to_csv(metadata_path, index=False, encoding="utf-8-sig")

## 5. 상품 URL 모으기

카테고리 첫 페이지에 표시된 순서대로 상품 주소를 모음.
실패한 상품이 생길 수 있어 목표보다 조금 더 많은 후보를 확보.
이미 목표 수만큼 저장되어 있으면 Chrome을 열지 않음.

이 셀에서 Chrome이 열리면 **다음 셀까지 실행**. 다음 셀이 끝나면 Chrome이 닫힘.


In [19]:
driver = None
product_urls = []

if len(completed_ids) < TARGET_PRODUCTS:
    driver = webdriver.Chrome()
    driver.set_page_load_timeout(60)
    wait = WebDriverWait(driver, 30)
    try:
        driver.get(CATEGORY_URL)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'article a[href^="/goods/"] h4')))

        for scroll_count in range(8):
            soup = BeautifulSoup(driver.page_source, "html.parser")
            for link in soup.select('article a[href^="/goods/"]'):
                product_url = urljoin(CATEGORY_URL, link["href"])
                if link.select_one("h4") and product_url not in product_urls:
                    product_urls.append(product_url)

            if len(product_urls) >= TARGET_PRODUCTS + 10:
                break
            driver.execute_script("window.scrollBy(0, 1000)")
            time.sleep(2)
    except Exception:
        driver.quit()
        driver = None
        raise
else:
    print("이미 목표 상품 수만큼 저장되어 있습니다.")

print("수집 후보:", len(product_urls), "상품")


이미 목표 상품 수만큼 저장되어 있습니다.
수집 후보: 0 상품


## 6. 대표·상세 이미지와 메타데이터 저장

대표 이미지는 상품 페이지 상단에서, 상세 이미지는 **상품설명 영역의 소개 사진**에서 가져옴.
기존 상세페이지와 새 상세페이지 형식을 모두 확인.
대표·상세가 같은 원본 주소이면 해당 상품을 건너뛰고 화면에 이유를 표시.

**`alt_exists`는 alt 속성 자체가 있으면 1, 없으면 0.**
각 상품의 두 이미지가 저장될 때마다 CSV도 저장.


In [20]:
detail_selector = '#description .goods_intro img, #description [data-block-type="INTRO"] img'
failed_products = []

try:
    for product_url in product_urls:
        if len(completed_ids) >= TARGET_PRODUCTS:
            break
        product_id = urlsplit(product_url).path.rstrip("/").split("/")[-1]
        if product_id in completed_ids:
            continue

        saved_files = []
        try:
            driver.get(product_url)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
            description = wait.until(EC.presence_of_element_located((By.ID, "description")))
            driver.execute_script("arguments[0].scrollIntoView();", description)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, detail_selector)))

            soup = BeautifulSoup(driver.page_source, "html.parser")
            product_name = soup.select_one("h1").get_text(strip=True)
            main_image = soup.select_one('#product-atf img[data-nimg="fill"]')
            detail_image = soup.select_one(detail_selector)
            if main_image is None or detail_image is None:
                raise ValueError("대표 또는 상세 이미지가 없습니다.")

            main_url = get_image_url(main_image, product_url)
            detail_url = get_image_url(detail_image, product_url)
            if image_key(main_url) == image_key(detail_url):
                raise ValueError("대표와 상세가 같은 이미지입니다.")

            main_content, main_extension = download_image(main_url)
            time.sleep(1)
            detail_content, detail_extension = download_image(detail_url)
            images = [
                (main_image, main_url, main_content, main_extension),
                (detail_image, detail_url, detail_content, detail_extension)
            ]
            product_rows = []
            crawl_date = datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="seconds")

            for position, (image_tag, image_url, content, extension) in enumerate(images, start=1):
                image_path = image_dir / f"kurly_{product_id}_{position}.{extension}"
                saved_files.append(image_path)
                image_path.write_bytes(content)
                product_rows.append({
                    "product_id": product_id, "product_name": product_name,
                    "category": CATEGORY, "product_url": product_url,
                    "image_id": len(rows) + position, "image_position": position,
                    "image_url": image_url, "original_alt": image_tag.get("alt", ""),
                    "alt_exists": int(bool(image_tag.get("alt", "").strip())), "crawl_date": crawl_date
                })

        except PermissionError:
            raise
        except Exception as error:
            for image_path in saved_files:
                image_path.unlink(missing_ok=True)
            failed_products.append({"product_id": product_id, "reason": str(error)})
            print("수집 실패:", product_id, error)
        else:
            rows.extend(product_rows)
            completed_ids.add(product_id)
            save_metadata()
            print(f"[{len(completed_ids)}/{TARGET_PRODUCTS}] {product_name}")
        time.sleep(3)
finally:
    if driver is not None:
        driver.quit()

save_metadata()
print("완료:", len(completed_ids), "상품 /", len(rows), "장")
if len(completed_ids) < TARGET_PRODUCTS:
    print("목표 수량에 미달했습니다. 위의 실패 메시지를 확인해 주세요.")


완료: 50 상품 / 100 장


## 7. 저장 결과 확인

50상품을 수집하면 CSV 100행과 이미지 100장이 생성.
`vegetables/images`에서 두 사진이 해당 상품의 대표·소개 사진인지 확인.
같은 장면의 다른 구도나 확대 사진이 포함될 수 있음.


In [21]:
data = pd.read_csv(metadata_path, dtype={"product_id": str}, keep_default_na=False)
print("상품 수:", data["product_id"].nunique())
print("CSV 행 수:", len(data))
print("이미지 파일 수:", len(list(image_dir.glob("kurly_*"))))
print("CSV 위치:", metadata_path.resolve())
display(data.head(6))
if failed_products:
    display(pd.DataFrame(failed_products))


상품 수: 50
CSV 행 수: 100
이미지 파일 수: 100
CSV 위치: D:\2026\Capstone2\Capstone-Design-Alt-Text\convenience_meals\convenience_meals_metadata.csv


,product_id,product_name,category,product_url,image_id,image_position,image_url,original_alt,alt_exists,crawl_date
0,1001207466,[차려낸] 햄 가득 송탄식 부대찌개,convenience_meals,https://www.kurly.com/goods/1001207466,1,1,https://product-image.kurly.com/hdims/resize/%...,상품-대표-이미지 1,1,2026-09-22T22:28:42+09:00
1,1001207466,[차려낸] 햄 가득 송탄식 부대찌개,convenience_meals,https://www.kurly.com/goods/1001207466,2,2,https://product-image.kurly.com/hdims/resize/%...,,0,2026-09-22T22:28:42+09:00
2,5156742,[이연복의 목란] 짬뽕 2인분 (맵기선택),convenience_meals,https://www.kurly.com/goods/5156742,3,1,https://product-image.kurly.com/hdims/resize/%...,상품-대표-이미지 1,1,2026-09-22T22:28:51+09:00
3,5156742,[이연복의 목란] 짬뽕 2인분 (맵기선택),convenience_meals,https://www.kurly.com/goods/5156742,4,2,https://img-cf.kurly.com/hdims/resize/%3E1010x...,,0,2026-09-22T22:28:51+09:00
4,1000572062,[사리원] 소불고기 전골,convenience_meals,https://www.kurly.com/goods/1000572062,5,1,https://product-image.kurly.com/hdims/resize/%...,상품-대표-이미지 1,1,2026-09-22T22:29:00+09:00
5,1000572062,[사리원] 소불고기 전골,convenience_meals,https://www.kurly.com/goods/1000572062,6,2,https://product-image.kurly.com/hdims/resize/%...,,0,2026-09-22T22:29:00+09:00
